In [10]:
from huggingface_hub import snapshot_download

repo_id = "official-stockfish/fishtest_pgns"
repo_type = "dataset"
allow_pattern = "19-*-*/*/*.pgn.gz"
local_dir = "./raw_dataset/"

snapshot_download(repo_id=repo_id, repo_type=repo_type, allow_patterns=allow_pattern, local_dir=local_dir)


Fetching ... files: 0it [00:00, ?it/s]

'C:\\Users\\yount\\Documents\\projets\\Pytorch NNUE\\data\\raw_dataset'

In [ ]:
import os
import glob
import gzip
import chess
import chess.pgn
import torch
import multiprocessing as mp
from tqdm import tqdm

# --- CONFIGURATION ---
INPUT_DIR = "./raw_dataset"       # Dossier racine contenant les YY-MM-DD/
OUTPUT_DIR = "./halfkp_data"   # Dossier de destination des .pt
CHUNK_SIZE = 500_000           # Nombre de positions par fichier .pt

# --- OPTIMISATIONS D'ACCÈS ---
# Index par type de pièce (0: None, 1: Pion=0, 2: Cavalier=1, 3: Fou=2, 4: Tour=3, 5: Dame=4)
# Permet un accès tableau en O(1) au lieu d'un dictionnaire
PIECE_OFFSET = [0, 0, 1, 2, 3, 4] 

def fast_halfkp(board):
    """
    Générateur HalfKP hautement optimisé (Bitwise operations & list lookups).
    """
    stm = board.turn
    king_sq = board.king(stm)
    
    # Symétrie ultra-rapide avec l'opérateur bit à bit XOR (56 = 0x38)
    if stm == chess.BLACK:
        king_sq ^= 56 

    indices = []
    # piece_map() est rapide en python-chess
    for sq, piece in board.piece_map().items():
        if piece.piece_type == chess.KING:
            continue
            
        p_idx = PIECE_OFFSET[piece.piece_type]
        if piece.color != stm:
            p_idx += 5
            
        # XOR 56 pour la symétrie de la case
        p_sq = sq if stm == chess.WHITE else sq ^ 56
        
        idx = king_sq * 640 + p_idx * 64 + p_sq
        indices.append(idx)
        
    return indices

def process_file_chunk(args):
    """
    Fonction exécutée par chaque cœur CPU indépendamment.
    """
    worker_id, file_paths = args
    
    print(f"Worker {worker_id} démarre avec {len(file_paths)} fichiers.")
    
    # Création du dossier de sortie par le worker principal
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    positions = []
    labels = []
    chunk_idx = 0
    
    # tqmd uniquement sur le worker 0 pour suivre l'avancement global
    iterator = tqdm(file_paths, position=worker_id, desc=f"Worker {worker_id}") if worker_id == 0 else file_paths

    for file_path in iterator:
        print(f"\nWorker {worker_id} traite: {file_path}")
        try:
            with gzip.open(file_path, "rt", encoding="utf-8") as f:
                while True:
                    game = chess.pgn.read_game(f)
                    if game is None: break
                    
                    # Parsage hyper rapide du résultat
                    res_str = game.headers.get("Result")
                    if res_str == "1-0": res_val = 1.0
                    elif res_str == "0-1": res_val = 0.0
                    elif res_str == "1/2-1/2": res_val = 0.5
                    else: continue # Ignore les parties inachevées ("*")
                    
                    board = game.board()
                    for move in game.mainline_moves():
                        # L'état avant le coup
                        idx_list = fast_halfkp(board)
                        label = res_val if board.turn == chess.WHITE else 1.0 - res_val
                        
                        # Stockage en int16 (2 octets par index au lieu de 8)
                        positions.append(torch.tensor(idx_list, dtype=torch.int8))
                        labels.append(label)
                        
                        board.push(move)
                        
                        # Sauvegarde et vidage de la RAM
                        if len(positions) >= CHUNK_SIZE:
                            save_path = os.path.join(OUTPUT_DIR, f"w{worker_id}_batch_{chunk_idx}.pt")
                            torch.save({
                                'indices': positions, 
                                'labels': torch.tensor(labels, dtype=torch.float32)
                            }, save_path)
                            positions = []
                            labels = []
                            chunk_idx += 1
                            
        except Exception as e:
            # Sécurité si un fichier gz est corrompu lors du téléchargement
            print(f"\nErreur sur le fichier {file_path}: {e}")
            
    # Sauvegarde du reliquat en fin de traitement
    if positions:
        save_path = os.path.join(OUTPUT_DIR, f"w{worker_id}_batch_{chunk_idx}.pt")
        torch.save({
            'indices': positions, 
            'labels': torch.tensor(labels, dtype=torch.float32)
        }, save_path)

if __name__ == '__main__':
    # 1. Lister tous les fichiers .pgn.gz
    print("Recherche des fichiers...")
    search_pattern = os.path.join(INPUT_DIR, "**", "*.pgn.gz")
    all_files = glob.glob(search_pattern, recursive=True)
    print(f"{len(all_files)} fichiers trouvés.")
    
    if len(all_files) == 0:
        print("Vérifiez le chemin INPUT_DIR.")
        exit()

    # 2. Répartir la charge sur les cœurs CPU
    num_cores = max(1, mp.cpu_count() - 1) # Garde 1 cœur pour le système
    print(f"Lancement du multiprocessing sur {num_cores} cœurs...")
    
    # Découpage de la liste des fichiers en N sous-listes
    chunked_files = [all_files[i::num_cores] for i in range(num_cores)]
    
    # Création des arguments pour chaque worker : (ID_worker, Liste_de_fichiers)
    worker_args = [(i, chunk) for i, chunk in enumerate(chunked_files)]
    
    print(f"Chaque worker traitera environ {len(all_files) // num_cores} fichiers.")
    # 3. Lancement !
    with mp.Pool(num_cores) as pool:
        pool.map(process_file_chunk, worker_args)
        
    print(f"\nConversion terminée ! Les fichiers .pt sont dans {OUTPUT_DIR}")

Recherche des fichiers...
265 fichiers trouvés.
Lancement du multiprocessing sur 7 cœurs...
Chaque worker traitera environ 37 fichiers.
